In [164]:
import geopandas as gpd
import pandas as pd
import numpy as np
import altair as alt
import folium
import requests

# **API Load of GeoJson Files**

Takes in files from [Sound IQ ](https://experience.arcgis.com/experience/5416ae7c51ae4244a1a42ba679c1479f/page/KELP) filtering them for the beds we care about.


Assings them to the kelp geo data frame (kelpgdf)

In [165]:
url = ('https://services.arcgis.com/6lCKYNJLvwTXqrmp/arcgis/rest/services/SIQ_Kelp_Surveys/FeatureServer/24/query?where=1%3D1&outFields=*&outSR=4326&f=geojson')
kelp_beds = gpd.read_file(url)

sites = ['Possession Point', 'Lowell Point', 'Hoypus Point', 'Polnell Point', "Ebey's Landing"]

kelpgdf = kelp_beds[kelp_beds['Location'].isin(sites)]
kelpgdf.head()

,OBJECTID,Survey_ID,County,Location,Bed_Name,site_code,Survey_Date,Survey_Day,Survey_Month,Survey_Year,...,ToWa,BeL,BeR,Notes,Extent_Start_Waypoint,Extent_End_Waypoint,Bulb_m2,Shape__Area,Shape__Length,geometry
0,1,Ebeys_Landing_June_14_2015,Island,Ebey's Landing,Ebey's Landing,EBL,2015-06-14,14,6,2015,...,None,None,None,None,None,None,1.7590,86346.953125,1834.903661,"POLYGON ((-122.70131 48.18503, -122.70127 48.1..."
1,2,Ebeys_Landing_July_19_2015,Island,Ebey's Landing,Ebey's Landing,EBL,2015-07-19,19,7,2015,...,None,None,None,None,None,None,1.8222,133068.851562,2287.171615,"POLYGON ((-122.7013 48.18498, -122.70122 48.18..."
2,3,Ebeys_Landing_Aug_15_2015,Island,Ebey's Landing,Ebey's Landing,EBL,2015-08-15,15,8,2015,...,None,None,None,None,None,None,1.2000,166985.789062,2866.943303,"POLYGON ((-122.7028 48.18701, -122.7027 48.186..."
3,4,Ebeys_Landing_Sept_26_2015,Island,Ebey's Landing,Ebey's Landing,EBL,2015-09-26,26,9,2015,...,None,None,None,None,None,None,1.6889,162725.648438,2677.089037,"POLYGON ((-122.70231 48.1864, -122.70226 48.18..."
37,38,Ebeys_Landing_June_25_2016_1,Island,Ebey's Landing,Ebey's Landing,EBL,2016-06-25,25,6,2016,...,None,None,None,None,None,None,NaN,166333.468750,2196.325725,"POLYGON ((-122.699 48.18254, -122.69915 48.182..."


In [166]:
kelpgdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 131 entries, 0 to 556
Data columns (total 60 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   OBJECTID                  131 non-null    int32         
 1   Survey_ID                 131 non-null    object        
 2   County                    131 non-null    object        
 3   Location                  131 non-null    object        
 4   Bed_Name                  131 non-null    object        
 5   site_code                 131 non-null    object        
 6   Survey_Date               131 non-null    datetime64[ms]
 7   Survey_Day                131 non-null    int32         
 8   Survey_Month              131 non-null    int32         
 9   Survey_Year               131 non-null    int32         
 10  Start_Time                126 non-null    object        
 11  End_Time                  119 non-null    object        
 12  NWSC_Max_Ext       

In [167]:
kelpgdf.value_counts('Location')

,count
Location,
Possession Point,30
Ebey's Landing,28
Hoypus Point,28
Lowell Point,23
Polnell Point,22


## **SoundIQ Data Cleaning**

We surely dont need this many columns and we dont want temperatures for all 4 data points so we can collapse those. (We may want to do something similar for depth? But depth isnt quite as complicated so.)

In [168]:
kelp_red = kelpgdf[[
    'Location',
    'site_code',
    'Survey_Date',
    'Survey_Day',
    'Survey_Month',
    'Survey_Year',
    'Acres_NWSC_clip',
    'Miles',
    'Temp',
    'Temp1_Shore_Edge',
    'Temp1_Water_Edge',
    'Temp2_Shore_Edge',
    'Temp2_Water_Edge',
    'Ave_Temp_Shore_Edge',
    'Ave_Temp_Water_Edge',
    'Depth1_Shore_Edge',
    'Depth1_Water_Edge',
    'Depth2_Shore_Edge',
    'Depth2_Water_Edge',
    'Ave_Depth_Shore_Edge_m',
    'Ave_Depth_Water_Edge_m']]

kelp_red.info()

<class 'pandas.core.frame.DataFrame'>
Index: 131 entries, 0 to 556
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Location                131 non-null    object        
 1   site_code               131 non-null    object        
 2   Survey_Date             131 non-null    datetime64[ms]
 3   Survey_Day              131 non-null    int32         
 4   Survey_Month            131 non-null    int32         
 5   Survey_Year             131 non-null    int32         
 6   Acres_NWSC_clip         131 non-null    float64       
 7   Miles                   131 non-null    float64       
 8   Temp                    51 non-null     float64       
 9   Temp1_Shore_Edge        71 non-null     object        
 10  Temp1_Water_Edge        70 non-null     object        
 11  Temp2_Shore_Edge        49 non-null     object        
 12  Temp2_Water_Edge        50 non-null     object        


### Temperature Averaging from Data Foundations

In [169]:
def prepare_temp(df):

    temp_cols = [
        "Temp",
        "Ave_Temp_Shore_Edge",
        "Ave_Temp_Water_Edge",
        "Temp1_Shore_Edge",
        "Temp2_Shore_Edge",
        "Temp1_Water_Edge",
        "Temp2_Water_Edge",
    ]

    for c in temp_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")


    df["Temp_Final"] = np.where(
        ~df["Temp"].isna(),
        df["Temp"],
        np.where(
            (~df["Ave_Temp_Shore_Edge"].isna()) &
            (~df["Ave_Temp_Water_Edge"].isna()),
            (df["Ave_Temp_Shore_Edge"] + df["Ave_Temp_Water_Edge"]) / 2,
            (
                ((df["Temp1_Shore_Edge"] + df["Temp2_Shore_Edge"]) / 2) +
                ((df["Temp1_Water_Edge"] + df["Temp2_Water_Edge"]) / 2)
            ) / 2
        )
    )

    return df.head()

prepare_temp(kelp_red)

/tmp/ipykernel_920/1948477887.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[c] = pd.to_numeric(df[c], errors="coerce")
/tmp/ipykernel_920/1948477887.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Temp_Final"] = np.where(


,Location,site_code,Survey_Date,Survey_Day,Survey_Month,Survey_Year,Acres_NWSC_clip,Miles,Temp,Temp1_Shore_Edge,...,Temp2_Water_Edge,Ave_Temp_Shore_Edge,Ave_Temp_Water_Edge,Depth1_Shore_Edge,Depth1_Water_Edge,Depth2_Shore_Edge,Depth2_Water_Edge,Ave_Depth_Shore_Edge_m,Ave_Depth_Water_Edge_m,Temp_Final
0,Ebey's Landing,EBL,2015-06-14,14,6,2015,9.490754,0.760311,11.0,NaN,...,NaN,NaN,NaN,0.8266,1.8053,None,None,NaN,NaN,11.0
1,Ebey's Landing,EBL,2015-07-19,19,7,2015,14.626063,0.947730,13.0,NaN,...,NaN,NaN,NaN,0.549,1.6714,None,None,NaN,NaN,13.0
2,Ebey's Landing,EBL,2015-08-15,15,8,2015,18.354044,1.188004,13.0,NaN,...,NaN,NaN,NaN,0.6924,1.6775,None,None,NaN,NaN,13.0
3,Ebey's Landing,EBL,2015-09-26,26,9,2015,17.885642,1.109370,12.0,NaN,...,NaN,NaN,NaN,0.3965,1.7995,None,None,NaN,NaN,12.0
37,Ebey's Landing,EBL,2016-06-25,25,6,2016,18.282310,0.909986,14.0,NaN,...,NaN,NaN,NaN,1.525,6.1,None,None,NaN,NaN,14.0


### Depth Averaging & Standardization

Some entries have only Avg Shore Depth / Avg Deep Depth. Some only have individual depths from each data point.

We want to plot the Avg Shore and Deep changes over time so we need to fill in those that are missing.

In [170]:
def prepare_shore_depth(df):

    depth_cols = [
        'Depth1_Shore_Edge',
        'Depth1_Water_Edge',
        'Depth2_Shore_Edge',
        'Depth2_Water_Edge',
        'Ave_Depth_Shore_Edge_m',
        'Ave_Depth_Water_Edge_m'
    ]

    for c in depth_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")


    df["Avg_ShoreDepth_Final"] = np.where(
        ~df['Ave_Depth_Shore_Edge_m'].isna(),
        df['Ave_Depth_Shore_Edge_m'],
        np.where(
            (~df['Depth1_Shore_Edge'].isna()) &
            (~df['Depth2_Shore_Edge'].isna()),
            (df['Depth1_Shore_Edge'] + df['Depth2_Shore_Edge']) / 2,
            (
                ((df['Depth1_Shore_Edge']
              )
            )
        )
    )
  )

    return df.head(100)



def prepare_water_depth(df):

    depth_cols = [
        'Depth1_Shore_Edge',
        'Depth1_Water_Edge',
        'Depth2_Shore_Edge',
        'Depth2_Water_Edge',
        'Ave_Depth_Shore_Edge_m',
        'Ave_Depth_Water_Edge_m'
    ]

    for c in depth_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")


    df["Avg_WaterDepth_Final"] = np.where(
        ~df['Ave_Depth_Water_Edge_m'].isna(),
        df['Ave_Depth_Water_Edge_m'],
        np.where(
            (~df['Depth1_Water_Edge'].isna()) &
            (~df['Depth2_Water_Edge'].isna()),
            (df['Depth1_Water_Edge'] + df['Depth2_Water_Edge']) / 2,
            (
                ((df['Depth1_Water_Edge']
              )
            )
        )
    )
  )

    return df.head(100)

In [171]:
prepare_shore_depth(kelp_red)
prepare_water_depth(kelp_red)

/tmp/ipykernel_920/519262331.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[c] = pd.to_numeric(df[c], errors="coerce")
/tmp/ipykernel_920/519262331.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Avg_ShoreDepth_Final"] = np.where(
/tmp/ipykernel_920/519262331.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/st

,Location,site_code,Survey_Date,Survey_Day,Survey_Month,Survey_Year,Acres_NWSC_clip,Miles,Temp,Temp1_Shore_Edge,...,Ave_Temp_Water_Edge,Depth1_Shore_Edge,Depth1_Water_Edge,Depth2_Shore_Edge,Depth2_Water_Edge,Ave_Depth_Shore_Edge_m,Ave_Depth_Water_Edge_m,Temp_Final,Avg_ShoreDepth_Final,Avg_WaterDepth_Final
0,Ebey's Landing,EBL,2015-06-14,14,6,2015,9.490754,0.760311,11.0,NaN,...,NaN,0.8266,1.8053,NaN,NaN,NaN,NaN,11.00,0.8266,1.8053
1,Ebey's Landing,EBL,2015-07-19,19,7,2015,14.626063,0.947730,13.0,NaN,...,NaN,0.5490,1.6714,NaN,NaN,NaN,NaN,13.00,0.5490,1.6714
2,Ebey's Landing,EBL,2015-08-15,15,8,2015,18.354044,1.188004,13.0,NaN,...,NaN,0.6924,1.6775,NaN,NaN,NaN,NaN,13.00,0.6924,1.6775
3,Ebey's Landing,EBL,2015-09-26,26,9,2015,17.885642,1.109370,12.0,NaN,...,NaN,0.3965,1.7995,NaN,NaN,NaN,NaN,12.00,0.3965,1.7995
37,Ebey's Landing,EBL,2016-06-25,25,6,2016,18.282310,0.909986,14.0,NaN,...,NaN,1.5250,6.1000,NaN,NaN,NaN,NaN,14.00,1.5250,6.1000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
450,Hoypus Point,HOYP,2023-08-15,15,8,2023,3.468347,0.424088,NaN,14.5,...,13.7,0.9100,5.4900,0.91,3.66,0.910,4.565,14.20,0.9100,4.5650
451,Hoypus Point,HOYP,2023-09-28,28,9,2023,2.464259,0.300851,NaN,12.5,...,12.4,1.3700,6.1000,1.37,5.49,1.290,5.705,12.45,1.2900,5.7050
452,Lowell Point,LWLP,2023-06-20,20,6,2023,1.001324,0.382484,NaN,13.9,...,13.9,1.1000,1.5200,1.01,1.31,-0.795,-0.430,13.90,-0.7950,-0.4300
453,Lowell Point,LWLP,2023-07-16,16,7,2023,4.359027,0.891510,NaN,16.5,...,15.7,1.1600,4.1100,1.80,5.97,2.645,5.415,15.75,2.6450,5.4150


### Slight Renaming for Clarity

In [172]:
kelp_clean= kelp_red[[
    'Location',
    'site_code',
    'Survey_Date',
    'Survey_Day',
    'Survey_Month',
    'Survey_Year',
    'Acres_NWSC_clip',
    'Miles',
    'Temp_Final',
    'Depth1_Shore_Edge',
    'Depth1_Water_Edge',
    'Depth2_Shore_Edge',
    'Depth2_Water_Edge',
    'Ave_Depth_Shore_Edge_m',
    'Ave_Depth_Water_Edge_m',
    'Avg_ShoreDepth_Final',
    'Avg_WaterDepth_Final']].rename(columns={
         'Acres_NWSC_clip' : 'area_acres',
         'Miles': 'perimeter_miles',
         'Temp_Final' : 'sstemp'
    })

kelp_clean['Location'] = kelp_clean['Location'].replace({
    "Ebey's Landing": 'Ebeys Landing'})

kelp_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 131 entries, 0 to 556
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Location                131 non-null    object        
 1   site_code               131 non-null    object        
 2   Survey_Date             131 non-null    datetime64[ms]
 3   Survey_Day              131 non-null    int32         
 4   Survey_Month            131 non-null    int32         
 5   Survey_Year             131 non-null    int32         
 6   area_acres              131 non-null    float64       
 7   perimeter_miles         131 non-null    float64       
 8   sstemp                  99 non-null     float64       
 9   Depth1_Shore_Edge       95 non-null     float64       
 10  Depth1_Water_Edge       93 non-null     float64       
 11  Depth2_Shore_Edge       47 non-null     float64       
 12  Depth2_Water_Edge       39 non-null     float64       


In [173]:
kelp_clean['Location'].unique()

array(['Ebeys Landing', 'Hoypus Point', 'Lowell Point', 'Polnell Point',
       'Possession Point'], dtype=object)

### Date Time Conversion

In [174]:
kelp_clean['Survey_Date'] = pd.to_datetime(kelp_clean['Survey_Date'])
kelp_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 131 entries, 0 to 556
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Location                131 non-null    object        
 1   site_code               131 non-null    object        
 2   Survey_Date             131 non-null    datetime64[ms]
 3   Survey_Day              131 non-null    int32         
 4   Survey_Month            131 non-null    int32         
 5   Survey_Year             131 non-null    int32         
 6   area_acres              131 non-null    float64       
 7   perimeter_miles         131 non-null    float64       
 8   sstemp                  99 non-null     float64       
 9   Depth1_Shore_Edge       95 non-null     float64       
 10  Depth1_Water_Edge       93 non-null     float64       
 11  Depth2_Shore_Edge       47 non-null     float64       
 12  Depth2_Water_Edge       39 non-null     float64       


In [175]:
kelp_clean.head()

,Location,site_code,Survey_Date,Survey_Day,Survey_Month,Survey_Year,area_acres,perimeter_miles,sstemp,Depth1_Shore_Edge,Depth1_Water_Edge,Depth2_Shore_Edge,Depth2_Water_Edge,Ave_Depth_Shore_Edge_m,Ave_Depth_Water_Edge_m,Avg_ShoreDepth_Final,Avg_WaterDepth_Final
0,Ebeys Landing,EBL,2015-06-14,14,6,2015,9.490754,0.760311,11.0,0.8266,1.8053,NaN,NaN,NaN,NaN,0.8266,1.8053
1,Ebeys Landing,EBL,2015-07-19,19,7,2015,14.626063,0.947730,13.0,0.5490,1.6714,NaN,NaN,NaN,NaN,0.5490,1.6714
2,Ebeys Landing,EBL,2015-08-15,15,8,2015,18.354044,1.188004,13.0,0.6924,1.6775,NaN,NaN,NaN,NaN,0.6924,1.6775
3,Ebeys Landing,EBL,2015-09-26,26,9,2015,17.885642,1.109370,12.0,0.3965,1.7995,NaN,NaN,NaN,NaN,0.3965,1.7995
37,Ebeys Landing,EBL,2016-06-25,25,6,2016,18.282310,0.909986,14.0,1.5250,6.1000,NaN,NaN,NaN,NaN,1.5250,6.1000


# **Locally Hosted Data Sets**

lat lon coordinates for all beds in MRC.

Conductivity, Temperature, & Dissolved Solid (CTD - Salinity/Temp) measurements from 5 beds in 2025 (Possession, Hoypus, Lowell, Ebeys (August only), Polnell (August Only).

In [176]:
locations = pd.read_csv("/content/latlon_sites.csv")
ctd_2025 = pd.read_csv("/content/kelp_bed_ctd_2025.csv")

## **Locations Cleaning**

All that needs to be done is dropping the ones we dont have the other data for, specifically ctd.

In [177]:
locations.head()

,Location,Review,lon,lat
0,Clallam Bay,No,-124.274586,48.256240
1,FWB2/Observatory Point,No,-123.637116,48.149835
2,FWB3/Extension,Review for 2024,-123.637779,48.151343
3,North Beach,No,-122.777051,48.144927
4,Edmonds Dive Park,No,-122.381880,47.816669


In [178]:
locations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Location  26 non-null     object 
 1   Review    25 non-null     object 
 2   lon       26 non-null     float64
 3   lat       26 non-null     float64
dtypes: float64(2), object(2)
memory usage: 964.0+ bytes


In [179]:
loc_sites = ['Possession Point', 'Lowell Point Camano State Park', 'Hoypus', 'Polnell', "Ebey's"]

clean_loc = locations[locations['Location'].isin(loc_sites)]

In [180]:
clean_loc['Location'] = clean_loc['Location'].replace({
    'Lowell Point Camano State Park': 'Lowell Point',
    'Hoypus': 'Hoypus Point',
    'Polnell': 'Pollnell Point',
    "Ebey's": 'Ebeys Landing'})

/tmp/ipykernel_920/189826202.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_loc['Location'] = clean_loc['Location'].replace({


In [181]:
clean_loc.head()

,Location,Review,lon,lat
10,Possession Point,No,-122.388517,47.901769
11,Ebeys Landing,No,-122.703190,48.183653
12,Pollnell Point,No,-122.565838,48.274563
13,Lowell Point,No,-122.485688,48.120390
14,Hoypus Point,No,-122.602076,48.410683


In [182]:
clean_loc.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5 entries, 10 to 14
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Location  5 non-null      object 
 1   Review    5 non-null      object 
 2   lon       5 non-null      float64
 3   lat       5 non-null      float64
dtypes: float64(2), object(2)
memory usage: 200.0+ bytes


## **CTD Cleaning**

Not any cleaning neccesary (I think), was built to not need cleaning.

In [183]:
ctd_2025.head()

,site,bed,date,data_point,temp,salinity,conductivity,depth
0,Ebeys Landing,Bed Main,2025-08-19,DE1,12.8,30.50998,36.04,0
1,Ebeys Landing,Bed Main,2025-08-19,DE1,12.8,30.68870,36.23,2
2,Ebeys Landing,Bed Main,2025-08-19,DE1,12.8,30.69811,36.24,4
3,Ebeys Landing,Bed Main,2025-08-19,DE1,12.7,30.77950,36.24,6
4,Ebeys Landing,Bed Main,2025-08-19,DE1,12.7,30.78894,36.25,8


In [184]:
ctd_2025.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 353 entries, 0 to 352
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   site          353 non-null    object 
 1   bed           353 non-null    object 
 2   date          353 non-null    object 
 3   data_point    353 non-null    object 
 4   temp          350 non-null    float64
 5   salinity      347 non-null    float64
 6   conductivity  280 non-null    float64
 7   depth         353 non-null    int64  
dtypes: float64(3), int64(1), object(4)
memory usage: 22.2+ KB


In [185]:
ctd_2025['site'].unique()

array(['Ebeys Landing', 'Hoypus Point', 'Lowell Point', 'Polnell Point',
       'Possession Point'], dtype=object)

# **EDA**

## Depth

### All

In [186]:
depth = alt.Chart(kelp_clean).transform_fold(
    ["Avg_ShoreDepth_Final", "Avg_WaterDepth_Final"],
    as_=["edge", "depth"]
    ).mark_line().encode(
        alt.X('yearmonth(Survey_Date):T', title=None),
        alt.Y('depth:Q', title = 'Depth(ft)'),
        alt.Color('edge:N'),
    ).properties(
    width=800,
    height=400)

depth

alt.Chart(...)

### Possession Point

In [187]:
depth_poss = alt.Chart(kelp_clean).transform_filter(
        alt.datum.Location == 'Possession Point'
    ).transform_fold(
        ["Avg_ShoreDepth_Final", "Avg_WaterDepth_Final"],
          as_=["edge", "depth"]
    ).transform_filter(
        alt.expr.isValid(alt.datum.depth)
    ).mark_line().encode(
        alt.X('yearmonth(Survey_Date):O', title=None),
        alt.Y('depth:Q', title = 'Depth(m)').scale(reverse = True),
        alt.Color('edge:N')
    ).properties(
    width=800,
    height=400)

depth_poss

alt.Chart(...)

In [188]:
kelp_clean.query("Location == 'Possession Point'")[['Survey_Date', 'Avg_ShoreDepth_Final', 'Avg_WaterDepth_Final', 'Depth1_Shore_Edge',
    'Depth1_Water_Edge',
    'Depth2_Shore_Edge',
    'Depth2_Water_Edge',
    'Ave_Depth_Shore_Edge_m',
    'Ave_Depth_Water_Edge_m']]

,Survey_Date,Avg_ShoreDepth_Final,Avg_WaterDepth_Final,Depth1_Shore_Edge,Depth1_Water_Edge,Depth2_Shore_Edge,Depth2_Water_Edge,Ave_Depth_Shore_Edge_m,Ave_Depth_Water_Edge_m
159,2017-08-24,2.8975,3.8125,2.8975,3.8125,NaN,NaN,NaN,NaN
220,2018-07-17,0.7000,4.3000,0.7000,4.3000,NaN,NaN,NaN,NaN
221,2018-08-14,1.8000,4.6000,1.8000,4.6000,NaN,NaN,NaN,NaN
222,2018-09-10,2.1000,5.5000,2.1000,5.5000,NaN,NaN,NaN,NaN
266,2019-06-21,1.9800,3.5100,1.9800,3.5100,NaN,NaN,NaN,NaN
267,2019-07-19,0.5800,4.3900,0.5800,4.3900,NaN,NaN,NaN,NaN
268,2019-08-16,1.6800,4.7200,1.6800,4.7200,NaN,NaN,NaN,NaN
269,2019-09-26,2.7400,5.7900,2.7400,5.7900,NaN,NaN,NaN,NaN
319,2020-06-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
320,2020-07-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Ebeys Landing

In [189]:
depth_ebey = alt.Chart(kelp_clean).transform_filter(
        alt.datum.Location == 'Ebeys Landing'
    ).transform_fold(
        ["Avg_ShoreDepth_Final", "Avg_WaterDepth_Final"],
          as_=["edge", "depth"]
    ).transform_filter(
        alt.expr.isValid(alt.datum.depth)
    ).mark_line().encode(
        alt.X('yearmonth(Survey_Date):O', title=None),
        alt.Y('depth:Q', title = 'Depth(m)').scale(reverse = True),
        alt.Color('edge:N')
    ).properties(
    width=800,
    height=400)

depth_ebey

alt.Chart(...)

In [190]:
kelp_clean.query("Location == 'Ebeys Landing'")[['Survey_Date', 'Avg_ShoreDepth_Final', 'Avg_WaterDepth_Final', 'Depth1_Shore_Edge',
    'Depth1_Water_Edge',
    'Depth2_Shore_Edge',
    'Depth2_Water_Edge',
    'Ave_Depth_Shore_Edge_m',
    'Ave_Depth_Water_Edge_m']]

,Survey_Date,Avg_ShoreDepth_Final,Avg_WaterDepth_Final,Depth1_Shore_Edge,Depth1_Water_Edge,Depth2_Shore_Edge,Depth2_Water_Edge,Ave_Depth_Shore_Edge_m,Ave_Depth_Water_Edge_m
0,2015-06-14,0.8266,1.8053,0.8266,1.8053,NaN,NaN,NaN,NaN
1,2015-07-19,0.5490,1.6714,0.5490,1.6714,NaN,NaN,NaN,NaN
2,2015-08-15,0.6924,1.6775,0.6924,1.6775,NaN,NaN,NaN,NaN
3,2015-09-26,0.3965,1.7995,0.3965,1.7995,NaN,NaN,NaN,NaN
37,2016-06-25,1.5250,6.1000,1.5250,6.1000,NaN,NaN,NaN,NaN
38,2016-07-22,1.6775,6.2525,1.6775,6.2525,NaN,NaN,NaN,NaN
39,2016-08-19,1.3725,7.3200,1.3725,7.3200,NaN,NaN,NaN,NaN
147,2017-06-26,-0.1525,5.3375,-0.1525,5.3375,NaN,NaN,NaN,NaN
148,2017-07-25,1.7538,6.3288,1.7538,6.3288,NaN,NaN,NaN,NaN
149,2017-08-23,1.3725,7.6250,1.3725,7.6250,NaN,NaN,NaN,NaN


## Ridgline Area

### Possession Point

In [191]:
area_poss = alt.Chart(kelp_clean).transform_filter(
        alt.datum.Location == 'Possession Point'
    ).mark_line().encode(
        alt.X('month(Survey_Date):O', title=None),
        alt.Y('area_acres:Q', title = 'Area(acre)'),
        alt.Color('year(Survey_Date):O')
    ).properties(
    width=800,
    height=400)

area_poss

alt.Chart(...)

In [192]:
step = 30
overlap = 1

alt.Chart(kelp_clean, height=step).properties(
    width=800
).transform_filter(
        alt.datum.Location == 'Possession Point'
).transform_timeunit(
    Month='month(Survey_Date)', Year='year(Survey_Date)'
).transform_aggregate(
   value='mean(area_acres)', groupby=['Year', 'Month']
).transform_joinaggregate(
    mean_area='mean(value)', groupby=['Year']
).mark_area(
    interpolate='monotone',
    fillOpacity=0.8,
    stroke='lightgray',
    strokeWidth=0.5
).encode(
    alt.X('Month:T', title = None),
    alt.Y(
        'value:Q',
        scale=alt.Scale(range=[step, -step * overlap]),
        axis=None
    ),
    alt.Fill(
        'mean_area:Q',
        legend=None,
        scale=alt.Scale(domain=[30, 5], scheme='redyellowblue')
    )
).facet(
    row=alt.Row(
        'Year:T',
        title=None,
        header=alt.Header(labelAngle=0, labelAlign='right', format='%Y')
    )
).properties(
    title='Acres Over Time',
    bounds='flush',
).configure_facet(
    spacing=0
).configure_view(
    stroke=None
).configure_title(
    anchor='end'
)

alt.FacetChart(...)

In [193]:
kelp_clean.query("Location == 'Possession Point'")[['Survey_Date', 'area_acres']]

,Survey_Date,area_acres
159,2017-08-24,21.484920
220,2018-07-17,15.344572
221,2018-08-14,38.002331
222,2018-09-10,36.006873
266,2019-06-21,0.181879
267,2019-07-19,57.367234
268,2019-08-16,60.670912
269,2019-09-26,60.457353
319,2020-06-26,11.679887
320,2020-07-17,24.652997


### Ebeys Landing

In [194]:
area_poss = alt.Chart(kelp_clean).transform_filter(
        alt.datum.Location == 'Ebeys Landing'
    ).mark_line().encode(
        alt.X('month(Survey_Date):O', title=None),
        alt.Y('area_acres:Q', title = 'Area(acre)'),
        alt.Color('year(Survey_Date):O')
    ).properties(
    width=800,
    height=400)

area_poss

alt.Chart(...)

In [195]:
step = 30
overlap = 1

alt.Chart(kelp_clean, height=step).properties(
    width=800
).transform_filter(
        alt.datum.Location == 'Ebeys Landing'
).transform_timeunit(
    Month='month(Survey_Date)', Year='year(Survey_Date)'
).transform_aggregate(
   value='mean(area_acres)', groupby=['Year', 'Month']
).transform_joinaggregate(
    mean_area='mean(value)', groupby=['Year']
).mark_area(
    interpolate='monotone',
    fillOpacity=0.8,
    stroke='lightgray',
    strokeWidth=0.5
).encode(
    alt.X('Month:T', title = None),
    alt.Y(
        'value:Q',
        scale=alt.Scale(range=[step, -step * overlap]),
        axis=None
    ),
    alt.Fill(
        'mean_area:Q',
        legend=None,
        scale=alt.Scale(domain=[30, 5], scheme='redyellowblue')
    )
).facet(
    row=alt.Row(
        'Year:T',
        title=None,
        header=alt.Header(labelAngle=0, labelAlign='right', format='%Y')
    )
).properties(
    title='Acres Over Time',
    bounds='flush',
).configure_facet(
    spacing=0
).configure_view(
    stroke=None
).configure_title(
    anchor='end'
)

alt.FacetChart(...)

In [196]:
kelp_clean.query("Location == 'Ebeys Landing'")[['Survey_Date', 'area_acres']]

,Survey_Date,area_acres
0,2015-06-14,9.490754
1,2015-07-19,14.626063
2,2015-08-15,18.354044
3,2015-09-26,17.885642
37,2016-06-25,18.282310
38,2016-07-22,20.396699
39,2016-08-19,18.985232
147,2017-06-26,14.044818
148,2017-07-25,20.161350
149,2017-08-23,20.945903


### All

In [197]:
step = 30
overlap = 1

alt.Chart(kelp_clean, height=step).properties(
    width=800
).transform_timeunit(
    Month='month(Survey_Date)', Year='year(Survey_Date)'
).transform_aggregate(
   value='mean(area_acres)', groupby=['Year', 'Month']
).transform_joinaggregate(
    mean_area='mean(value)', groupby=['Year']
).mark_area(
    interpolate='monotone',
    fillOpacity=0.8,
    stroke='lightgray',
    strokeWidth=0.5
).encode(
    alt.X('Month:T', title = None),
    alt.Y(
        'value:Q',
        scale=alt.Scale(range=[step, -step * overlap]),
        axis=None
    ),
    alt.Fill(
        'mean_area:Q',
        legend=None,
        scale=alt.Scale(domain=[30, 5], scheme='redyellowblue')
    )
).facet(
    row=alt.Row(
        'Year:T',
        title=None,
        header=alt.Header(labelAngle=0, labelAlign='right', format='%Y')
    )
).properties(
    title='Acres Over Time',
    bounds='flush',
).configure_facet(
    spacing=0
).configure_view(
    stroke=None
).configure_title(
    anchor='end'
)


alt.FacetChart(...)

## Locations Mapped -- Ahrial

In [198]:
# king county
item_id = "ff8f49238a7040ac9efcb65bf708b24d"
meta_url = f"https://kingcounty.maps.arcgis.com/sharing/rest/content/items/{item_id}?f=json"
service_url = requests.get(meta_url).json()["url"]

# counties
query_url = f"{service_url}/0/query?where=1%3D1&outFields=*&outSR=4326&f=geojson"
geojson_data = requests.get(query_url).json()

# initlize on washington
kelpbed_map = folium.Map(
    location=[47.50, -120.74],
    zoom_start=7,
    tiles="https://arcgisonline.com{z}/{y}/{x}",
    attr="Esri",
    width=700,
    height=480
)

# counties
folium.GeoJson(
    geojson_data,
    style_function=lambda feature: {
        "fillColor": "#e8ecef",
        "color": "#4a5568",
        "weight": 0.8,
        "fillOpacity": 0.5,
    }
).add_to(kelpbed_map)

# kelp beds
for _, row in locations.iterrows():
    tooltip_text = (
        f"<b>Location:</b> {row['Location']}<br>"
        f"<b>Latitude:</b> {row['lat']:.4f}<br>"
        f"<b>Longitude:</b> {row['lon']:.4f}"
    )
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=6,
        color="darkgreen",
        fill=True,
        fill_color="darkgreen",
        fill_opacity=0.85,
        tooltip=folium.Tooltip(tooltip_text)
    ).add_to(kelpbed_map)

# map
kelpbed_map


### Just Ebey's landing

In [199]:
ebey_loc = locations[locations['Location'] == "Ebey's"]
ebey_loc

,Location,Review,lon,lat
11,Ebey's,No,-122.70319,48.183653


In [200]:
target_lon = float(ebey_loc['lon'].iloc[0])
target_lat = float(ebey_loc['lat'].iloc[0])

background = alt.Chart(alt.Data(values=geojson_data["features"])).mark_geoshape(
    fill='#e8ecef',
    stroke='#4a5568',
    strokeWidth=0.8
)

points = alt.Chart(ebey_loc).mark_circle(
    color='#2ca02c',
    stroke='#1b661b',
    strokeWidth=1.5,
    size=100,
    opacity=1.0
).encode(
    longitude='lon:Q',
    latitude='lat:Q',
    tooltip=[
        alt.Tooltip('Location:N', title='Location'),
        alt.Tooltip('lat:Q', title='Latitude', format='.4f'),
        alt.Tooltip('lon:Q', title='Longitude', format='.4f')
    ]
)


ebey_map = (background + points).project(
    type='mercator',
    center=[target_lon, target_lat],
    scale=25000
).properties(
    width=700,
    height=480
)

ebey_map


alt.LayerChart(...)

### Map (lat-lon + ctd)

### Use the geometry to map the kelp beds from year to year

In [201]:
kelp_geo = kelpgdf.copy()

In [202]:
def prepare_temp(df):

    temp_cols = [
        "Temp",
        "Ave_Temp_Shore_Edge",
        "Ave_Temp_Water_Edge",
        "Temp1_Shore_Edge",
        "Temp2_Shore_Edge",
        "Temp1_Water_Edge",
        "Temp2_Water_Edge",
    ]

    for c in temp_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")


    df["Temp_Final"] = np.where(
        ~df["Temp"].isna(),
        df["Temp"],
        np.where(
            (~df["Ave_Temp_Shore_Edge"].isna()) &
            (~df["Ave_Temp_Water_Edge"].isna()),
            (df["Ave_Temp_Shore_Edge"] + df["Ave_Temp_Water_Edge"]) / 2,
            (
                ((df["Temp1_Shore_Edge"] + df["Temp2_Shore_Edge"]) / 2) +
                ((df["Temp1_Water_Edge"] + df["Temp2_Water_Edge"]) / 2)
            ) / 2
        )
    )

    return df.head()

prepare_temp(kelp_geo)

,OBJECTID,Survey_ID,County,Location,Bed_Name,site_code,Survey_Date,Survey_Day,Survey_Month,Survey_Year,...,BeL,BeR,Notes,Extent_Start_Waypoint,Extent_End_Waypoint,Bulb_m2,Shape__Area,Shape__Length,geometry,Temp_Final
0,1,Ebeys_Landing_June_14_2015,Island,Ebey's Landing,Ebey's Landing,EBL,2015-06-14,14,6,2015,...,None,None,None,None,None,1.7590,86346.953125,1834.903661,"POLYGON ((-122.70131 48.18503, -122.70127 48.1...",11.0
1,2,Ebeys_Landing_July_19_2015,Island,Ebey's Landing,Ebey's Landing,EBL,2015-07-19,19,7,2015,...,None,None,None,None,None,1.8222,133068.851562,2287.171615,"POLYGON ((-122.7013 48.18498, -122.70122 48.18...",13.0
2,3,Ebeys_Landing_Aug_15_2015,Island,Ebey's Landing,Ebey's Landing,EBL,2015-08-15,15,8,2015,...,None,None,None,None,None,1.2000,166985.789062,2866.943303,"POLYGON ((-122.7028 48.18701, -122.7027 48.186...",13.0
3,4,Ebeys_Landing_Sept_26_2015,Island,Ebey's Landing,Ebey's Landing,EBL,2015-09-26,26,9,2015,...,None,None,None,None,None,1.6889,162725.648438,2677.089037,"POLYGON ((-122.70231 48.1864, -122.70226 48.18...",12.0
37,38,Ebeys_Landing_June_25_2016_1,Island,Ebey's Landing,Ebey's Landing,EBL,2016-06-25,25,6,2016,...,None,None,None,None,None,NaN,166333.468750,2196.325725,"POLYGON ((-122.699 48.18254, -122.69915 48.182...",14.0


In [203]:
kelp_geo['Location'] = kelp_geo['Location'].replace({
    "Ebey's Landing": 'Ebeys Landing'})

In [204]:
kelp_geo['Survey_Date'] = pd.to_datetime(kelp_geo['Survey_Date'])

In [205]:
merged_max_acre = pd.merge(max_acres, kelp_geo, on=['site_code','Survey_Date'], how="inner")

In [206]:
cols_to_keep = [
    "site_code",
    "Location_x",
    "Bed_Name",
    "County",
    "Survey_Date",
    "Survey_Year_x",
    "area_acres",
    "perimeter_miles",
    "Temp_Final",
    "Tidal_Ht_m",
    "Tide_Station",
    "Weather",
    "Observations",
    "geometry",
]

clean_gdf = merged_max_acre[cols_to_keep].copy()

# rename columns
clean_gdf = clean_gdf.rename(
    columns={"Location_x": "Location", "Survey_Year_x": "Survey_Year"}
)

In [207]:
clean_gdf

,site_code,Location,Bed_Name,County,Survey_Date,Survey_Year,area_acres,perimeter_miles,Temp_Final,Tidal_Ht_m,Tide_Station,Weather,Observations,geometry
0,EBL,Ebeys Landing,Ebey's Landing,Island,2015-08-15,2015,18.354044,1.188004,13.0000,NaN,Admiralty Head,None,None,"POLYGON ((-122.7028 48.18701, -122.7027 48.186..."
1,EBL,Ebeys Landing,Ebey's Landing,Island,2016-07-22,2016,20.396699,1.163997,12.0000,NaN,Admiralty Head 12:30;-1.5,None,None,"POLYGON ((-122.70382 48.18777, -122.70373 48.1..."
2,EBL,Ebeys Landing,Ebey's Landing,Island,2017-09-16,2017,21.534816,0.966515,11.0000,NaN,Admiralty Head (9447905),None,1 sea lion cruising outer perimeter of bed 1 W...,"POLYGON ((-122.70193 48.18563, -122.70192 48.1..."
3,EBL,Ebeys Landing,Ebey's Landing,Island,2018-08-07,2018,21.293413,1.262661,12.0000,-0.3109,Admiralty Head,Clear,"Sori abundant, few bryozoans, kelp crabs absen...","POLYGON ((-122.70402 48.1868, -122.704 48.1867..."
4,EBL,Ebeys Landing,Ebey's Landing,Island,2019-08-13,2019,41.131035,1.629254,13.0000,-0.2286,Admiralty Head (9447905),Clouds,"12 Heermans gulls in bed, feeding; ~ 12 other ...","POLYGON ((-122.69749 48.18024, -122.69751 48.1..."
5,EBL,Ebeys Landing,Ebey's Landing,Island,2020-08-30,2020,41.613652,1.565123,NaN,-0.3760,Admiralty Head (9447905),Clear,None,"POLYGON ((-122.69712 48.1803, -122.69712 48.18..."
6,EBL,Ebeys Landing,Ebey's Landing,Island,2020-08-30,2020,41.613652,1.565123,NaN,-0.3760,Admiralty Head (9447905),Clear,None,"POLYGON ((-122.69712 48.1803, -122.69712 48.18..."
7,EBL,Ebeys Landing,Ebey's Landing,Island,2021-08-11,2021,37.681341,1.646982,NaN,NaN,Admiralty Head (9447905),Clear,"1. Numerous herring schools throughout bed, r...","POLYGON ((-122.69601 48.18075, -122.69607 48.1..."
8,EBL,Ebeys Landing,Ebey's Landing,Island,2022-08-23,2022,38.629474,1.598878,12.1500,-0.0610,Admiralty Head (9447905),Fog_mist,> 100 gulls (including Heerman's gulls) in fee...,"POLYGON ((-122.69629 48.18059, -122.69635 48.1..."
9,EBL,Ebeys Landing,Ebey's Landing,Island,2023-08-14,2023,30.708477,1.992345,12.0000,NaN,Admiralty Head (9447905),Clear,Sori abundant (bulbs & blades look good & at v...,"MULTIPOLYGON (((-122.69945 48.18207, -122.6993..."


In [208]:
ebey_max = clean_gdf[clean_gdf['Location'] == 'Ebeys Landing']

### Max extent of Ebey from year to year

In [209]:
# survey date as string
gdf = gpd.GeoDataFrame(ebey_max, geometry='geometry', crs='EPSG:4326').copy()
gdf['Survey_Date'] = gdf['Survey_Date'].astype(str)
gdf['Survey_Year'] = gdf['Survey_Year'].astype(str)

# map for Polygon and MultiPolygon
m = folium.Map(location=[48.275, -122.565], tiles='cartodbpositron', zoom_start=14)

folium.GeoJson(
    gdf,
    tooltip=folium.GeoJsonTooltip(fields=['Bed_Name', 'Survey_Year', 'area_acres'])
).add_to(m)

m.fit_bounds(m.get_bounds())
m


/usr/local/lib/python3.13/dist-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


## Trend Analysis of kelp and regression

In [210]:
idx = kelp_clean.groupby(['Location','Survey_Year'])['area_acres'].idxmax()
# keep the entire row
max_acres = kelp_clean.loc[idx]

max_acres

,Location,site_code,Survey_Date,Survey_Day,Survey_Month,Survey_Year,area_acres,perimeter_miles,sstemp,Depth1_Shore_Edge,Depth1_Water_Edge,Depth2_Shore_Edge,Depth2_Water_Edge,Ave_Depth_Shore_Edge_m,Ave_Depth_Water_Edge_m,Avg_ShoreDepth_Final,Avg_WaterDepth_Final
2,Ebeys Landing,EBL,2015-08-15,15,8,2015,18.354044,1.188004,13.0000,0.6924,1.6775,NaN,NaN,NaN,NaN,0.69240,1.6775
38,Ebeys Landing,EBL,2016-07-22,22,7,2016,20.396699,1.163997,12.0000,1.6775,6.2525,NaN,NaN,NaN,NaN,1.67750,6.2525
150,Ebeys Landing,EBL,2017-09-16,16,9,2017,21.534816,0.966515,11.0000,1.0675,7.3200,NaN,NaN,NaN,NaN,1.06750,7.3200
210,Ebeys Landing,EBL,2018-08-07,7,8,2018,21.293413,1.262661,12.0000,2.6000,7.0000,NaN,NaN,NaN,NaN,2.60000,7.0000
260,Ebeys Landing,EBL,2019-08-13,13,8,2019,41.131035,1.629254,13.0000,0.3800,5.4100,NaN,NaN,NaN,NaN,0.38000,5.4100
312,Ebeys Landing,EBL,2020-08-30,30,8,2020,41.613652,1.565123,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
359,Ebeys Landing,EBL,2021-08-11,11,8,2021,37.681341,1.646982,NaN,6.0960,4.2672,5.4864,NaN,NaN,NaN,5.79120,4.2672
405,Ebeys Landing,EBL,2022-08-23,23,8,2022,38.629474,1.598878,12.1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
447,Ebeys Landing,EBL,2023-08-14,14,8,2023,30.708477,1.992345,12.0000,1.5200,8.2300,1.0700,7.6200,-1.040,5.585,-1.04000,5.5850
489,Ebeys Landing,EBL,2024-08-20,20,8,2024,32.205306,1.394123,11.3500,4.8800,6.4000,1.2200,6.7100,NaN,NaN,3.05000,6.5550


## Perimeter/Area ratio

In [211]:
alt.Chart(kelp_clean).transform_calculate(
    PA='datum.perimeter_miles / datum.area_acres'
).mark_circle(opacity=0.6).encode(
    x=alt.X('area_acres:Q',
            scale=alt.Scale(type='log', domain=[0.1, 1000]),
            title='Area (acres)',
            axis=alt.Axis(values=[0.1, 1, 10, 100, 1000], format='~g')),
    y=alt.Y('PA:Q',
            scale=alt.Scale(type='log', domain=[0.01, 1]),
            title='Perimeter / Area (mi/acre)',
            axis=alt.Axis(values=[0.01, 0.1, 1], format='~g')),
    color=alt.Color('Location:N', scale=alt.Scale(scheme='category10'))
).properties(
    title='Kelp Perimeter-to-Area Ratio vs. Area Log Scale'
).interactive()

alt.Chart(...)

Acres and Perimeter are both quantitative values, and are therefore the most visually effective when encoding by position. Locations are categorical values, and therefore can be effectively encoded with color. I used a log scale to capture a wide range of values. From this plot, we can see that the perimeter/area ratio decreases as area decreases.

##Coefficient of Variance

In [212]:
cv_df = kelp_clean.groupby('Location')['area_acres'].agg(['mean', 'std']).reset_index()
cv_df['CV'] = cv_df['std'] / cv_df['mean']

base = alt.Chart(cv_df).encode(x=alt.X('Location:N', title=None))

boxes = base.mark_rect().encode(
    color=alt.Color('CV:Q', scale=alt.Scale(scheme='reds'))
)

text = base.mark_text(fontSize=16, fontWeight='bold').encode(
    text=alt.Text('CV:Q', format='.2f')
)

(boxes + text).properties(width=500, height=80, title='CV Score by Location')

alt.LayerChart(...)

An increased coefficient of variance implies that the acreage has greater variation. A sequential-single hue color scheme and text encodings encodes the CV variable by showing the concentration in color and quantitative values by text. Polnell point has an increased CV score compared to other locations.

## Pycnocline (Salinity and Temperature Changes with Depth)

In [215]:
alt.Chart(ctd_2025).mark_boxplot().encode(
    alt.X('month(date)'),
    alt.Y('temp:Q').scale(zero=False)
)

alt.Chart(...)

In [217]:
alt.Chart(ctd_2025).mark_boxplot().encode(
    alt.X('month(date)'),
    alt.Y('salinity:Q').scale(zero=False)
)

alt.Chart(...)

In [219]:
alt.Chart(ctd_2025).mark_line(color='firebrick').encode(
    alt.Color('month(date)').title('Survey Month'),
    alt.Y('depth').title('depth').scale(reverse = True),
    alt.X('mean(temp)').title('Temperature (C)').scale(reverse = True),
)

alt.Chart(...)

In [220]:
alt.Chart(ctd_2025).mark_line(color='firebrick').encode(
    alt.Color('month(date)').title('Survey Month'),
    alt.Y('depth').title('depth').scale(reverse = True),
    alt.X('mean(salinity)').title('Salinity (ppt)'),
)

alt.Chart(...)

In [221]:
alt.Chart(ctd_2025).mark_bar().encode(
    alt.X('month(date)').title('Survey Month'),
    alt.Y('depth').bin(step=2).title('Depth (ft)').scale(reverse = True),
    alt.Color(('mean(temp)'), bin=True).title('Mean Temperature (C)'),
    alt.Column('data_point')
)

alt.Chart(...)